# 12 — Segmentation Model Training (Stage 3)

Full model validation for the three-experiment design:

| Experiment | Input | Channels | Purpose |
|---|---|---|---|
| **A** | Single-date (Jul 30) | 9 | Conventional baseline |
| **B** | 4 phenological dates (Jan, Mar, Jul, Nov) | 36 | Multi-temporal naive |
| **C** | Stage 2 CNN forward-selection output | K* | Proposed method |

Each experiment × 2 architectures (DeepLabV3+CBAM, SegFormer) = **6 total runs**.

**Prerequisites**: run `02_image_processing.ipynb` and `04_feature_analysis.ipynb` first.

---

In [ ]:
import sys, os, re, time
from glob import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import rasterio
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"  # must be set before import
import mlflow

sys.path.append("../")

from geoai.geoai.train import RasterPatchDataset, train_semantic_one_epoch
from geoai.geoai.utils.device import get_device
from src.models import DeepLabV3PlusCBAM, build_segformer

DEVICE = get_device()
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = os.path.abspath("../")
PROCESSED_DIR    = os.path.join(PROJECT_ROOT, "data/processed")
S2_PROCESSED_DIR = os.path.join(PROCESSED_DIR, "s2")
# Per-year CDL paths (year-based train/test split)
CDL_BY_YEAR = {
    "2022": os.path.join(PROCESSED_DIR, "cdl/cdl_2022_study_area_filtered.tif"),
    "2023": os.path.join(PROCESSED_DIR, "cdl/cdl_2023_study_area_filtered.tif"),
    "2024": os.path.join(PROCESSED_DIR, "cdl/cdl_2024_study_area_filtered.tif"),
}
TRAIN_YEARS = ["2022", "2023"]   # combined for Stage 3 training
TEST_YEAR   = "2024"             # held-out test year
# Legacy alias for class-weight computation (use 2022 as reference)
CDL_FILTERED = CDL_BY_YEAR["2022"]
MODELS_DIR       = os.path.join(PROJECT_ROOT, "ml_models")
STAGE2_CSV       = os.path.join(PROCESSED_DIR, "stage2_history.csv")

os.makedirs(MODELS_DIR, exist_ok=True)


# ── S2 band metadata ──────────────────────────────────────────────────────────
# GEE export order: B1, B2 … B7, B8, B8A, B11, B12  (11 bands)
S2_BAND_NAMES    = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
N_BANDS_PER_DATE = len(S2_BAND_NAMES)

# 9 vegetation bands used for Exp A and B (excludes coastal B1 and redundant B8A)
VEGE_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]

# ── CDL classes ───────────────────────────────────────────────────────────────
KEEP_CLASSES = [3, 6, 24, 36, 37, 54, 69, 75, 76, 220]
CLASS_NAMES  = {
    3:   "Rice",        6:   "Sunflower",    24:  "Winter Wheat",
    36:  "Alfalfa",     37:  "Other Hay",     54:  "Tomatoes",
    69:  "Grapes",        75:  "Almonds",
    76:  "Walnuts",     220: "Plums",
}
CLASS_REMAP  = {cls_id: i + 1 for i, cls_id in enumerate(KEEP_CLASSES)}
NUM_CLASSES  = len(KEEP_CLASSES) + 1   # 11 (10 active crops + 0=background; Fallow moved to background)

REMAP_LUT = np.zeros(256, dtype=np.int64)
for cdl_id, model_id in CLASS_REMAP.items():
    if cdl_id < 256:
        REMAP_LUT[cdl_id] = model_id

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "https://mlflow-geoai.stelarea.com"
MLFLOW_EXPERIMENT   = "cropmap_segmentation_s2"

# ── Training hyperparameters ─────────────────────────────────────────────────
PATCH_SIZE     = 256     # full resolution
STRIDE         = 256     # non-overlapping
MIN_VALID_FRAC = 0.3
BATCH_SIZE     = 8
MAX_EPOCHS     = 100
EARLY_STOP     = 10
VAL_FRAC       = 0.15    # fraction of train-year patches held out as val
SEED           = 42

# Year-based split: TRAIN_YEARS patches → train+val; TEST_YEAR → test
# No random patch split across years (temporal honesty)

# Per-architecture hyperparameters (from methodology.tex)
ARCH_CFG = {
    "deeplabv3plus_cbam": {"lr": 1e-4,  "weight_decay": 1e-4,  "encoder": "resnet50"},
    "segformer":          {"lr": 6e-5,  "weight_decay": 1e-2,  "encoder": "mit_b2"},
}

# ── Verify paths ─────────────────────────────────────────────────────────────
# All processed S2 files — split by year at dataset-creation time
S2_PROCESSED = sorted(glob(f"{S2_PROCESSED_DIR}/*_processed.tif"))
print(f"S2 processed files : {len(S2_PROCESSED)}")
print(f"CDL filtered       : {os.path.exists(CDL_FILTERED)}")
for p in S2_PROCESSED:
    print(f"  {os.path.basename(p)}")
print(f"\nConfig: PATCH_SIZE={PATCH_SIZE}, MAX_EPOCHS={MAX_EPOCHS}, EARLY_STOP={EARLY_STOP}")


## [Demo] Pre-Chip Rasters to Disk

Pre-extract 256×256 patch pairs (S2 image + CDL mask) and save them as `.npz` files.
Useful for:
- **Visual inspection** of what the model actually sees
- **Fast iteration** when experimenting locally (no rasterio overhead per batch)
- **Sharing** a small representative subset without the full TIF

> **Note**: The production pipeline uses `RasterPatchDataset` (on-the-fly) and does
> **not** write chips to disk — this section is for exploration / demo only.
> Chips are saved to `data/processed/demo_chips/`.

In [ ]:
import shutil
from torch.utils.data import Dataset


def chip_rasters(
    s2_paths, cdl_path, band_indices, output_dir,
    patch_size=256, stride=256, min_valid_frac=0.3,
    remap_lut=None, nodata=-9999.0, max_chips=None, tag="",
    dry_run=False,
):
    """
    Pre-chip multi-temporal S2 + CDL rasters to disk as .npz files.

    Each .npz contains:
        image : float32 (C, patch_size, patch_size)  per-channel [0,1] normalised
        mask  : int64   (patch_size, patch_size)      remapped class IDs

    Only patches where the fraction of crop pixels >= min_valid_frac are saved.

    Args:
        dry_run  : if True, count valid patches and estimate storage without writing files.
                   Returns (n_valid, n_total, estimated_bytes).
        max_chips: stop after saving/counting this many chips (None = all)
        tag      : optional prefix for filenames (e.g. year)

    Returns (dry_run=False): n_saved (int)
    Returns (dry_run=True) : (n_valid, n_total, estimated_bytes)
    """
    if not dry_run:
        os.makedirs(output_dir, exist_ok=True)

    # ── Stack all S2 bands into memory ────────────────────────────────────────
    arrays = []
    for p in s2_paths:
        with rasterio.open(p) as src:
            arr = src.read().astype(np.float32)
        arr[arr == nodata] = np.nan
        arrays.append(arr)
    stacked = np.concatenate(arrays, axis=0)  # (total_C, H, W)

    # ── Load CDL ──────────────────────────────────────────────────────────────
    with rasterio.open(cdl_path) as src:
        cdl = src.read(1).astype(np.int32)

    if remap_lut is not None:
        cdl_clipped = np.clip(cdl, 0, len(remap_lut) - 1)
        cdl = remap_lut[cdl_clipped].astype(np.int64)

    H, W  = cdl.shape
    ps    = patch_size
    n_ch  = len(band_indices)

    # raw uncompressed bytes per chip: C × H × W × 4 (float32 image) + H × W × 8 (int64 mask)
    bytes_per_chip_raw = n_ch * ps * ps * 4 + ps * ps * 8
    # npz compression ratio empirically ~0.35–0.55 for normalised float32 patches
    COMPRESS_RATIO = 0.45
    bytes_per_chip_est = int(bytes_per_chip_raw * COMPRESS_RATIO)

    n_total = 0
    n_valid = 0

    for r in range(0, H - ps + 1, stride):
        for c in range(0, W - ps + 1, stride):
            n_total += 1
            mask = cdl[r:r+ps, c:c+ps]
            valid_frac = (mask > 0).sum() / (ps * ps)
            if valid_frac < min_valid_frac:
                continue

            n_valid += 1

            if not dry_run:
                img = stacked[np.array(band_indices, dtype=int)][:, r:r+ps, c:c+ps]
                img = np.where(np.isnan(img), 0.0, img)
                for ch in range(img.shape[0]):
                    mn, mx = img[ch].min(), img[ch].max()
                    img[ch] = (img[ch] - mn) / (mx - mn + 1e-9)
                prefix = f"{tag}_" if tag else ""
                fname  = os.path.join(output_dir, f"{prefix}chip_{n_valid:05d}.npz")
                np.savez_compressed(fname, image=img.astype(np.float32),
                                           mask=mask.astype(np.int64))

            if max_chips and n_valid >= max_chips:
                if dry_run:
                    return n_valid, n_total, n_valid * bytes_per_chip_est
                return n_valid

    if dry_run:
        return n_valid, n_total, n_valid * bytes_per_chip_est
    return n_valid


class ChipDataset(Dataset):
    """
    Load pre-chipped .npz patch pairs from a directory.
    Compatible with any num_workers value (plain file I/O, no rasterio).
    """
    def __init__(self, chip_dir):
        self.paths = sorted(glob(os.path.join(chip_dir, "*.npz")))
        assert self.paths, f"No .npz files found in {chip_dir}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        return torch.from_numpy(d["image"]), torch.from_numpy(d["mask"])


print("chip_rasters() and ChipDataset defined.")


In [ ]:
# ── Demo: chip one S2 file fully to disk ─────────────────────────────────────
# Uses the Jul 30 2022 file (Exp A — 9 vegetation bands).
# Runs chip_rasters() without a cap so we get the real patch count and disk size.

DEMO_CHIP_DIR = os.path.join(PROCESSED_DIR, "demo_chips")

# pick the single Jul-30 file
july30_file = next(
    (p for p in S2_PROCESSED
     if os.path.basename(p).split("_")[1] == "2022"
     and re.search(r"_2022_07_(29|30)_", p)),
    None,
)
assert july30_file, "Jul 30 2022 file not found in S2_PROCESSED"
print(f"Source file : {os.path.basename(july30_file)}")
print(f"Bands used  : {len(exp_A_idx)} channels (Exp A vegetation bands)")
print(f"Patch size  : {PATCH_SIZE}×{PATCH_SIZE}  stride={STRIDE}")

if os.path.isdir(DEMO_CHIP_DIR):
    shutil.rmtree(DEMO_CHIP_DIR)

t0 = time.time()
n_saved = chip_rasters(
    s2_paths       = [july30_file],
    cdl_path       = CDL_FILTERED,
    band_indices   = exp_A_idx,
    output_dir     = DEMO_CHIP_DIR,
    patch_size     = PATCH_SIZE,
    stride         = STRIDE,
    min_valid_frac = MIN_VALID_FRAC,
    remap_lut      = REMAP_LUT,
    tag            = "2022_jul30",
)
elapsed = time.time() - t0

chip_files   = sorted(glob(os.path.join(DEMO_CHIP_DIR, "*.npz")))
total_bytes  = sum(os.path.getsize(p) for p in chip_files)
print(f"\nChips saved : {n_saved:,}")
print(f"Total size  : {total_bytes/1e6:.1f} MB")
print(f"Avg per chip: {total_bytes/n_saved/1e3:.1f} KB")
print(f"Time        : {elapsed:.1f}s")

# ── Visualise a sample grid of chips ─────────────────────────────────────────
CROP_COLORS_HEX = [
    "#000000", "#1E90FF", "#FFD700", "#8B4513", "#98FB98",
    "#A9A9A9", "#FF6347", "#800080", "#FF8C00", "#228B22", "#DDA0DD",
]
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
cmap = ListedColormap(CROP_COLORS_HEX)
norm = BoundaryNorm(range(NUM_CLASSES + 1), NUM_CLASSES)

# sample 8 evenly spaced chips
step       = max(1, n_saved // 8)
sample     = chip_files[::step][:8]
R_IDX, G_IDX, B_IDX = 2, 1, 0   # B4/B3/B2 within exp_A_idx

fig, axes = plt.subplots(2, len(sample), figsize=(3*len(sample), 6))
fig.suptitle(
    f"Sample chips — {os.path.basename(july30_file)}  "
    f"| {n_saved} valid patches  | {total_bytes/1e6:.0f} MB\n"
    f"top: RGB (B4/B3/B2)   bottom: CDL mask",
    fontsize=10,
)
for col, path in enumerate(sample):
    d   = np.load(path)
    img = d["image"]
    msk = d["mask"]
    rgb = np.clip(np.stack([img[R_IDX], img[G_IDX], img[B_IDX]], axis=-1), 0, 1)
    axes[0, col].imshow(rgb)
    axes[0, col].axis("off")
    axes[0, col].set_title(f"#{chip_files.index(path)}", fontsize=8)
    axes[1, col].imshow(msk, cmap=cmap, norm=norm, interpolation="nearest")
    axes[1, col].axis("off")

class_names_ordered = ["Background"] + [CLASS_NAMES[k] for k in KEEP_CLASSES]
patches = [mpatches.Patch(color=CROP_COLORS_HEX[i], label=class_names_ordered[i])
           for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc="lower center", ncol=6, fontsize=7,
           bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.show()


In [ ]:
# ── Load Stage 2 results ──────────────────────────────────────────────────────
stage2_bands   = None
stage2_run_id  = None

if os.path.exists(STAGE2_CSV):
    hist_df      = pd.read_csv(STAGE2_CSV)
    stage2_bands = hist_df[hist_df["accepted"] == True]["band"].tolist()
    print(f"Stage 2 — {len(stage2_bands)} bands selected:")
    for b in stage2_bands:
        print(f"  {b}")
else:
    print(f"\u26a0\ufe0f  {STAGE2_CSV} not found. Experiment C will be skipped.")

# ── Build LOCAL band indices (relative to a single year's file list) ──────────
# All years share the same dates (same MMDD, different year prefix).
# Per-year local indices are directly reusable for any _s2_for_year(yr) list.

def parse_date(path):
    m = re.search(r"_(\d{4})_(\d{2})_(\d{2})_processed", os.path.basename(path))
    return f"{m.group(1)}{m.group(2)}{m.group(3)}" if m else None

# Use the year with the most processed files as reference
by_year = {}
for p in S2_PROCESSED:
    yr = os.path.basename(p).split("_")[1]
    by_year.setdefault(yr, []).append(p)

ref_yr    = max(by_year, key=lambda y: len(by_year[y]))
ref_files = sorted(by_year[ref_yr])

local_band_names  = []
local_date_to_idx = {}   # date_str -> position within ref_yr file list
for i, p in enumerate(ref_files):
    d = parse_date(p)
    local_date_to_idx[d] = i
    local_band_names.extend([f"{b}_{d}" for b in S2_BAND_NAMES])

local_band_to_idx = {n: i for i, n in enumerate(local_band_names)}
available_dates   = sorted(local_date_to_idx.keys())
mmdd_to_date      = {d[4:]: d for d in available_dates}   # e.g. "0730" -> "20220730"

print(f"Local channels (ref year {ref_yr}): {len(local_band_names)}  "
      f"({len(ref_files)} dates x {N_BANDS_PER_DATE} bands)")
print(f"Available dates: {available_dates}")

# ── Exp A: best available single date ────────────────────────────────────────
july30_key = next(
    (k for k in available_dates if k[4:6] == "07" and k[6:8] in ("29", "30")),
    None,
)
if july30_key is None:
    july30_key = available_dates[-1]
    print(f"Jul 30 not available - using fallback date: {july30_key}")

file_off_A  = local_date_to_idx[july30_key] * N_BANDS_PER_DATE
exp_A_idx   = [file_off_A + S2_BAND_NAMES.index(b) for b in VEGE_BANDS]
exp_A_names = [f"{b}_{july30_key}" for b in VEGE_BANDS]
print(f"\nExp A - date: {july30_key}, channels: {len(exp_A_idx)}")

# ── Exp B: phenological dates x 9 vege bands ─────────────────────────────────
phenol_targets = {"Jan": [1], "Mar": [3], "Jul": [7, 8], "Nov": [11, 10]}
phenol_map     = {}
for label, months in phenol_targets.items():
    match = next(
        (d for d in available_dates if int(d[4:6]) in months),
        None,
    )
    if match:
        phenol_map[label] = match

if len(phenol_map) < 2:
    phenol_map = {f"D{i}": d for i, d in enumerate(available_dates)}
    print(f"Fewer than 2 phenological months - using all {len(available_dates)} dates")

exp_B_idx, exp_B_names = [], []
for _label, d in phenol_map.items():
    off = local_date_to_idx[d] * N_BANDS_PER_DATE
    exp_B_idx   += [off + S2_BAND_NAMES.index(b) for b in VEGE_BANDS]
    exp_B_names += [f"{b}_{d}" for b in VEGE_BANDS]

seen, dedup_idx, dedup_names = set(), [], []
for idx, name in zip(exp_B_idx, exp_B_names):
    if idx not in seen:
        seen.add(idx)
        dedup_idx.append(idx)
        dedup_names.append(name)
exp_B_idx, exp_B_names = dedup_idx, dedup_names
print(f"Exp B - dates: {phenol_map}, channels: {len(exp_B_idx)}")

# ── Exp C: Stage 2 selected bands (mapped via MMDD to local indices) ──────────
if stage2_bands:
    exp_C_idx, exp_C_names = [], []
    for band_name in stage2_bands:
        # "B11_20220101" -> b="B11", mmdd="0101"
        b, ds = band_name.rsplit("_", 1)
        mmdd = ds[4:]
        local_date = mmdd_to_date.get(mmdd)
        if local_date and b in S2_BAND_NAMES:
            local_name = f"{b}_{local_date}"
            idx = local_band_to_idx.get(local_name)
            if idx is not None:
                exp_C_idx.append(idx)
                exp_C_names.append(local_name)
    if not exp_C_idx:
        print("No Stage 2 bands match current processed files - Exp C skipped.")
        exp_C_idx = exp_C_names = None
    else:
        print(f"Exp C - channels: {len(exp_C_idx)}")
else:
    exp_C_idx   = None
    exp_C_names = None


In [ ]:
# ── Compute inverse-frequency class weights from CDL raster ──────────────────
with rasterio.open(CDL_FILTERED) as src:
    cdl_arr = src.read(1).astype(np.int32)

class_counts = np.zeros(NUM_CLASSES, dtype=np.float64)
class_counts[0] = (cdl_arr == 0).sum()   # background
for cdl_id, model_id in CLASS_REMAP.items():
    class_counts[model_id] += (cdl_arr == cdl_id).sum()

freq          = class_counts / (class_counts.sum() + 1e-9)
class_weights = 1.0 / (freq + 1e-9)
class_weights /= class_weights.sum()   # normalise so weights sum to 1
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights (inverse-frequency, normalised):")
print(f"  [0] Background: count={class_counts[0]:10.0f}  weight={class_weights[0]:.5f}")
for i, cdl_id in enumerate(KEEP_CLASSES, start=1):
    name = CLASS_NAMES.get(cdl_id, f"Class {cdl_id}")
    print(f"  [{i:2d}] {name:20s}: count={class_counts[i]:10.0f}  weight={class_weights[i]:.5f}")

In [ ]:
# ── Training helpers ──────────────────────────────────────────────────────────

def compute_miou(logits, labels, num_classes, ignore_index=0):
    """mIoU over foreground classes (class 0 skipped)."""
    preds  = logits.argmax(dim=1).view(-1).cpu().numpy()
    labels = labels.view(-1).cpu().numpy()
    ious = []
    for cls in range(1, num_classes):
        p = (preds == cls)
        l = (labels == cls)
        inter = (p & l).sum()
        union = (p | l).sum()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0


def compute_per_class_iou(logits, labels, num_classes):
    """Per-class IoU dict {model_class_id: iou}. Background (0) excluded."""
    preds  = logits.argmax(dim=1).view(-1).numpy()
    labels = labels.view(-1).numpy()
    ious = {}
    for cls in range(1, num_classes):
        p = (preds == cls)
        l = (labels == cls)
        inter = (p & l).sum()
        union = (p | l).sum()
        ious[cls] = float(inter / union) if union > 0 else float("nan")
    return ious


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, num_classes):
    """Run validation, return dict with loss, miou, oa."""
    model.eval()
    total_loss   = 0.0
    all_logits   = []
    all_labels   = []

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits       = model(imgs)
        total_loss  += criterion(logits, masks).item()
        all_logits.append(logits.cpu())
        all_labels.append(masks.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    preds = all_logits.argmax(dim=1)
    oa    = (preds == all_labels).float().mean().item()
    miou  = compute_miou(all_logits, all_labels, num_classes)

    return {
        "loss": total_loss / len(loader),
        "miou": miou,
        "oa":   oa,
    }


@torch.no_grad()
def evaluate_test_set(model, loader, num_classes):
    """Full test-set evaluation: per-class IoU, mIoU, OA."""
    model.eval()
    all_logits = []
    all_labels = []

    for imgs, masks in loader:
        logits = model(imgs.to(DEVICE))
        all_logits.append(logits.cpu())
        all_labels.append(masks.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return {
        "miou":          compute_miou(all_logits, all_labels, num_classes),
        "oa":            (all_logits.argmax(1) == all_labels).float().mean().item(),
        "per_class_iou": compute_per_class_iou(all_logits, all_labels, num_classes),
    }

In [ ]:
# ── Model builder ─────────────────────────────────────────────────────────────

def build_model(arch, in_channels, num_classes):
    cfg = ARCH_CFG[arch]
    if arch == "deeplabv3plus_cbam":
        model = DeepLabV3PlusCBAM(
            encoder_name=cfg["encoder"],
            encoder_weights="imagenet",
            in_channels=in_channels,
            num_classes=num_classes,
        )
    elif arch == "segformer":
        model = build_segformer(
            encoder_name=cfg["encoder"],
            encoder_weights="imagenet",
            in_channels=in_channels,
            num_classes=num_classes,
        )
    else:
        raise ValueError(f"Unknown architecture: {arch}")

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {arch} ({cfg['encoder']}): {n_params:,} parameters")
    return model.to(DEVICE)


# ── Main training runner ──────────────────────────────────────────────────────

def run_experiment(
    exp_name,
    arch,
    band_indices,
    band_names_list,
    description,
    stage2_run_id=None,
):
    """
    Train one experiment configuration and log everything to MLflow.

    Returns dict with experiment summary metrics.
    """
    cfg         = ARCH_CFG[arch]
    in_channels = len(band_indices)
    exp_dir     = os.path.join(MODELS_DIR, exp_name)
    best_ckpt   = os.path.join(exp_dir, "best_model.pth")
    os.makedirs(exp_dir, exist_ok=True)

    print(f"\n{'='*65}")
    print(f" {exp_name}")
    print(f"  arch={arch}  in_channels={in_channels}")
    print(f"  {description}")
    print(f"{'='*65}\n")

    # ── Year-based dataset split ──────────────────────────────────────────────
    # Train/val: all patches from TRAIN_YEARS (2022 + 2023), each paired with
    #            its own year's CDL label.
    # Test     : all patches from TEST_YEAR  (2024), paired with 2024 CDL.
    # This temporal split avoids leakage and tests true year-to-year generalisation.
    from torch.utils.data import ConcatDataset

    def _s2_for_year(yr):
        return sorted([p for p in S2_PROCESSED if os.path.basename(p).split("_")[1] == yr])

    # Build one dataset per training year, then concatenate
    train_year_datasets = []
    for yr in TRAIN_YEARS:
        yr_s2  = _s2_for_year(yr)
        yr_cdl = CDL_BY_YEAR[yr]
        if not yr_s2 or not os.path.exists(yr_cdl):
            print(f"  ⚠️  Skipping train year {yr}: {'no S2' if not yr_s2 else 'CDL missing'}")
            continue
        ds = RasterPatchDataset(
            s2_paths=yr_s2, cdl_path=yr_cdl,
            patch_size=PATCH_SIZE, stride=STRIDE,
            keep_classes=KEEP_CLASSES, remap_lut=REMAP_LUT,
            min_valid_frac=MIN_VALID_FRAC, band_indices=band_indices,
        )
        train_year_datasets.append(ds)
        print(f"  [{yr}] {len(ds):,} patches")

    assert train_year_datasets, "No training data found for any TRAIN_YEAR"
    train_val_ds = ConcatDataset(train_year_datasets)

    # Val split from combined train years
    n_total = len(train_val_ds)
    n_val   = max(1, int(VAL_FRAC * n_total))
    n_train = n_total - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = random_split(train_val_ds, [n_train, n_val], generator=gen)

    # Test: all patches from TEST_YEAR
    test_s2  = _s2_for_year(TEST_YEAR)
    test_cdl = CDL_BY_YEAR[TEST_YEAR]
    assert test_s2 and os.path.exists(test_cdl), f"Test year {TEST_YEAR} data missing"
    test_ds = RasterPatchDataset(
        s2_paths=test_s2, cdl_path=test_cdl,
        patch_size=PATCH_SIZE, stride=STRIDE,
        keep_classes=KEEP_CLASSES, remap_lut=REMAP_LUT,
        min_valid_frac=MIN_VALID_FRAC, band_indices=band_indices,
    )

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"  Patches: {n_train:,} train / {n_val:,} val / {len(test_ds):,} test ({TEST_YEAR})")

    # ── Model + optimiser + scheduler + loss ──────────────────────────────────
    model     = build_model(arch, in_channels, NUM_CLASSES)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=MAX_EPOCHS, power=0.9
    )
    criterion = nn.CrossEntropyLoss(
        weight=CLASS_WEIGHTS_TENSOR.to(DEVICE)
    )

    # ── MLflow run ────────────────────────────────────────────────────────────
    os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

    with mlflow.start_run(run_name=f"{exp_name}_{timestamp}") as run:
        mlflow.log_params({
            "experiment":     exp_name,
            "architecture":   arch,
            "encoder":        cfg["encoder"],
            "in_channels":    in_channels,
            "num_classes":    NUM_CLASSES,
            "patch_size":     PATCH_SIZE,
            "batch_size":     BATCH_SIZE,
            "max_epochs":     MAX_EPOCHS,
            "early_stopping": EARLY_STOP,
            "learning_rate":  cfg["lr"],
            "weight_decay":   cfg["weight_decay"],
            "optimizer":      "AdamW",
            "lr_scheduler":   "PolynomialLR(power=0.9)",
            "loss":           "WeightedCrossEntropy",
            "train_years":    str(TRAIN_YEARS),
            "test_year":      TEST_YEAR,
            "train_patches":  n_train,
            "val_patches":    n_val,
            "test_patches":   len(test_ds),
            "description":    description,
            "keep_classes":   str(KEEP_CLASSES),
        })
        mlflow.set_tag("band_names", str(band_names_list))
        mlflow.set_tag("n_bands",    str(in_channels))
        if stage2_run_id:
            mlflow.set_tag("stage2_run_id", stage2_run_id)

        # ── Training loop ─────────────────────────────────────────────────────
        best_miou  = 0.0
        no_improve = 0
        history    = []
        t_start    = time.time()

        for epoch in range(MAX_EPOCHS):
            t_ep = time.time()

            train_loss = train_semantic_one_epoch(
                model, optimizer, train_dl, DEVICE, epoch, criterion,
                print_freq=len(train_dl),   # print once per epoch
                verbose=False,
            )
            val_m   = validate_one_epoch(model, val_dl, criterion, DEVICE, NUM_CLASSES)
            scheduler.step()

            ep_t = time.time() - t_ep
            mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss":   val_m["loss"],
                "val_miou":   val_m["miou"],
                "val_oa":     val_m["oa"],
                "lr":         scheduler.get_last_lr()[0],
            }, step=epoch)

            history.append({
                "epoch":      epoch + 1,
                "train_loss": round(train_loss,       4),
                "val_loss":   round(val_m["loss"],    4),
                "val_miou":   round(val_m["miou"],    4),
                "val_oa":     round(val_m["oa"],      4),
                "epoch_t_s":  round(ep_t,              1),
            })

            if val_m["miou"] > best_miou:
                best_miou  = val_m["miou"]
                no_improve = 0
                torch.save({
                    "epoch":            epoch,
                    "model_state_dict": model.state_dict(),
                    "best_miou":        best_miou,
                    "band_indices":     band_indices,
                    "band_names":       band_names_list,
                    "in_channels":      in_channels,
                    "num_classes":      NUM_CLASSES,
                    "architecture":     arch,
                }, best_ckpt)
            else:
                no_improve += 1

            total_min = (time.time() - t_start) / 60
            print(
                f"  Ep {epoch+1:3d}/{MAX_EPOCHS} "
                f"loss={train_loss:.4f} val_loss={val_m['loss']:.4f} "
                f"mIoU={val_m['miou']:.4f} best={best_miou:.4f} "
                f"patience={no_improve}/{EARLY_STOP} "
                f"{ep_t:.0f}s  total={total_min:.1f}min"
            )

            if no_improve >= EARLY_STOP:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

        # ── Test evaluation ───────────────────────────────────────────────────
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        test_r = evaluate_test_set(model, test_dl, NUM_CLASSES)

        mlflow.log_metrics({
            "best_val_miou": best_miou,
            "test_miou":     test_r["miou"],
            "test_oa":       test_r["oa"],
            "total_epochs":  len(history),
        })

        # Per-class IoU
        for cls_id, iou in test_r["per_class_iou"].items():
            if not np.isnan(iou):
                name = CLASS_NAMES.get(KEEP_CLASSES[cls_id - 1], f"cls{cls_id}")
                mlflow.log_metric(
                    f"test_iou_{name.lower().replace('/', '_').replace(' ', '_')}",
                    iou,
                )

        # ── Artifacts ─────────────────────────────────────────────────────────
        hist_df  = pd.DataFrame(history)
        hist_csv = os.path.join(exp_dir, "training_history.csv")
        hist_df.to_csv(hist_csv, index=False)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(hist_df["epoch"], hist_df["train_loss"], label="Train")
        ax1.plot(hist_df["epoch"], hist_df["val_loss"],   label="Val")
        ax1.set(xlabel="Epoch", ylabel="Loss", title=f"{exp_name} — Loss")
        ax1.legend(); ax1.grid(True)

        ax2.plot(hist_df["epoch"], hist_df["val_miou"], color="green", label="Val mIoU")
        ax2.axhline(best_miou, linestyle="--", color="gray", label=f"Best={best_miou:.4f}")
        ax2.set(xlabel="Epoch", ylabel="mIoU", title=f"{exp_name} — mIoU")
        ax2.legend(); ax2.grid(True)

        plt.tight_layout()
        curve_path = os.path.join(exp_dir, "training_curve.png")
        plt.savefig(curve_path, dpi=150)
        plt.show()

        mlflow.log_artifact(best_ckpt)
        mlflow.log_artifact(hist_csv)
        mlflow.log_artifact(curve_path)

        run_id = run.info.run_id

    print(f"\n✅ {exp_name}  val_mIoU={best_miou:.4f}  test_mIoU={test_r['miou']:.4f}  run={run_id}")

    return {
        "exp_name":      exp_name,
        "arch":          arch,
        "in_channels":   in_channels,
        "best_val_miou": round(best_miou,      4),
        "test_miou":     round(test_r["miou"], 4),
        "test_oa":       round(test_r["oa"],   4),
        "total_epochs":  len(history),
        "run_id":        run_id,
        "ckpt":          best_ckpt,
    }

---
## Dataset Distribution — Training and Test Split

In [ ]:
PIXEL_AREA_HA = (10 * 10) / 10_000   # 10 m resolution → 0.01 ha per pixel

def count_class_pixels(cdl_path):
    """Return {model_id: pixel_count} for all 11 classes from one CDL raster."""
    with rasterio.open(cdl_path) as src:
        arr = src.read(1).astype(np.int32)
    counts = {}
    # Background: everything NOT in KEEP_CLASSES
    bg_mask = np.ones(arr.shape, dtype=bool)
    for cdl_id in KEEP_CLASSES:
        bg_mask &= (arr != cdl_id)
    counts[0] = int(bg_mask.sum())
    for cdl_id in KEEP_CLASSES:
        counts[CLASS_REMAP[cdl_id]] = int((arr == cdl_id).sum())
    return counts

# ── Pixel counts per year ──────────────────────────────────────────────────────
counts_by_year = {}
for yr, path in CDL_BY_YEAR.items():
    if os.path.exists(path):
        counts_by_year[yr] = count_class_pixels(path)
    else:
        print(f"⚠️  CDL not found: {path}")

# ── Aggregate train (2022+2023) and test (2024) ────────────────────────────────
train_counts = {
    mid: sum(counts_by_year.get(yr, {}).get(mid, 0) for yr in TRAIN_YEARS)
    for mid in range(NUM_CLASSES)
}
test_counts = counts_by_year.get(TEST_YEAR, {mid: 0 for mid in range(NUM_CLASSES)})

train_total = sum(train_counts.values())
test_total  = sum(test_counts.values())

# ── Build DataFrame ────────────────────────────────────────────────────────────
model_id_to_name = {0: "Background (+ Fallow)"}
model_id_to_name.update({CLASS_REMAP[k]: v for k, v in CLASS_NAMES.items()})

rows = []
for mid in range(NUM_CLASSES):
    tr = train_counts[mid]
    te = test_counts.get(mid, 0)
    rows.append({
        "Kelas"          : model_id_to_name[mid],
        "Train (px)"     : f"{tr:,}",
        "Train (ha)"     : f"{tr * PIXEL_AREA_HA:,.0f}",
        "Train (%)"      : f"{100 * tr / train_total:.1f}",
        "Test (px)"      : f"{te:,}",
        "Test (ha)"      : f"{te * PIXEL_AREA_HA:,.0f}",
        "Test (%)"       : f"{100 * te / test_total:.1f}",
    })

# totals row
rows.append({
    "Kelas"     : "Total",
    "Train (px)": f"{train_total:,}",
    "Train (ha)": f"{train_total * PIXEL_AREA_HA:,.0f}",
    "Train (%)" : "100.0",
    "Test (px)" : f"{test_total:,}",
    "Test (ha)" : f"{test_total * PIXEL_AREA_HA:,.0f}",
    "Test (%)"  : "100.0",
})

df_dist = pd.DataFrame(rows)
display(df_dist)

# ── Render as publication-style figure ────────────────────────────────────────
FIGURES_DIR_LOCAL = os.path.join(PROJECT_ROOT, "documents/thesis/figures")
os.makedirs(FIGURES_DIR_LOCAL, exist_ok=True)

cols     = list(df_dist.columns)
n_rows   = len(df_dist)
col_w    = [2.8, 1.1, 1.1, 0.9, 1.1, 1.1, 0.9]   # inches per column
fig_w    = sum(col_w) + 0.4
row_h    = 0.30
fig_h    = row_h * (n_rows + 2) + 0.6

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

tbl = ax.table(
    cellText  = df_dist.values,
    colLabels = cols,
    cellLoc   = "center",
    loc       = "center",
    colWidths = [w / fig_w for w in col_w],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.35)

# ── Styling: header row ────────────────────────────────────────────────────────
for col_i in range(len(cols)):
    cell = tbl[0, col_i]
    cell.set_facecolor("#2c3e50")
    cell.set_text_props(color="white", fontweight="bold")
    cell.set_edgecolor("white")

# ── Styling: data rows (zebra stripes, bold totals) ───────────────────────────
for row_i in range(1, n_rows + 1):
    is_total = (row_i == n_rows)
    bg = "#ecf0f1" if row_i % 2 == 0 else "white"
    for col_i in range(len(cols)):
        cell = tbl[row_i, col_i]
        cell.set_facecolor("#dfe6e9" if is_total else bg)
        cell.set_edgecolor("#bdc3c7")
        if is_total:
            cell.set_text_props(fontweight="bold")
        # left-align the class name column
        if col_i == 0:
            cell.set_text_props(ha="left")
            cell._loc = "left"

ax.set_title(
    f"Distribusi Dataset — Train ({', '.join(TRAIN_YEARS)}) vs Test ({TEST_YEAR})\n"
    f"(resolusi 10 m/piksel, 1 piksel = 0.01 ha)",
    fontsize=10, fontweight="bold", pad=12, loc="left",
)

plt.tight_layout()
savepath = os.path.join(FIGURES_DIR_LOCAL, "dataset_distribution.png")
plt.savefig(savepath, dpi=200, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {savepath}")
print(f"\nTrain years : {TRAIN_YEARS}  →  {train_total:,} px  ({train_total * PIXEL_AREA_HA:,.0f} ha)")
print(f"Test year   : {TEST_YEAR}   →  {test_total:,} px  ({test_total * PIXEL_AREA_HA:,.0f} ha)")

## Experiment A — Single-Date Baseline

9 vegetation bands from **Jul 30** (peak season, 99.6% valid coverage).  
Represents the conventional single-date approach without temporal information.

In [ ]:
results_A_dlv3 = run_experiment(
    exp_name     = "exp_A_deeplabv3plus_cbam",
    arch         = "deeplabv3plus_cbam",
    band_indices = exp_A_idx,
    band_names_list = exp_A_names,
    description  = f"Single-date {july30_key}, 9ch, DeepLabV3+CBAM",
)

In [ ]:
results_A_sfm = run_experiment(
    exp_name     = "exp_A_segformer",
    arch         = "segformer",
    band_indices = exp_A_idx,
    band_names_list = exp_A_names,
    description  = f"Single-date {july30_key}, 9ch, SegFormer",
)

## Experiment B — Multi-Temporal Naive Baseline

9 vegetation bands × 4 phenological dates = **36 channels**.  
Dates chosen by calendar (Jan, Mar, Jul, Nov) — no selection algorithm.

In [ ]:
results_B_dlv3 = run_experiment(
    exp_name     = "exp_B_deeplabv3plus_cbam",
    arch         = "deeplabv3plus_cbam",
    band_indices = exp_B_idx,
    band_names_list = exp_B_names,
    description  = f"4 phenological dates {list(phenol_map.values())}, 36ch, DeepLabV3+CBAM",
)

In [ ]:
results_B_sfm = run_experiment(
    exp_name     = "exp_B_segformer",
    arch         = "segformer",
    band_indices = exp_B_idx,
    band_names_list = exp_B_names,
    description  = f"4 phenological dates {list(phenol_map.values())}, 36ch, SegFormer",
)

## Experiment C — Multi-Temporal with Band Selection (Proposed Method)

**K\*** bands from Stage 2 CNN forward selection.  
Stage 1 determined order; Stage 2 determined the optimal cut-off.

In [ ]:
if exp_C_idx is None:
    print("⚠️  Experiment C skipped — run 04_feature_analysis.ipynb to generate Stage 2 results.")
else:
    results_C_dlv3 = run_experiment(
        exp_name        = "exp_C_deeplabv3plus_cbam",
        arch            = "deeplabv3plus_cbam",
        band_indices    = exp_C_idx,
        band_names_list = exp_C_names,
        description     = f"Stage2 selection K*={len(exp_C_idx)}ch, DeepLabV3+CBAM",
        stage2_run_id   = stage2_run_id,
    )

In [ ]:
if exp_C_idx is None:
    print("⚠️  Experiment C skipped.")
else:
    results_C_sfm = run_experiment(
        exp_name        = "exp_C_segformer",
        arch            = "segformer",
        band_indices    = exp_C_idx,
        band_names_list = exp_C_names,
        description     = f"Stage2 selection K*={len(exp_C_idx)}ch, SegFormer",
        stage2_run_id   = stage2_run_id,
    )

## Results Summary

In [ ]:
# Collect all results that were actually run
all_results = []
for var_name in [
    "results_A_dlv3", "results_A_sfm",
    "results_B_dlv3", "results_B_sfm",
    "results_C_dlv3", "results_C_sfm",
]:
    r = globals().get(var_name)
    if r is not None:
        all_results.append(r)

summary_df = pd.DataFrame(all_results)[[
    "exp_name", "arch", "in_channels",
    "best_val_miou", "test_miou", "test_oa", "total_epochs", "run_id"
]]
summary_df = summary_df.sort_values("test_miou", ascending=False)

print("\n=== Experiment Results Summary ===")
print(summary_df.to_string(index=False))

summary_csv = os.path.join(MODELS_DIR, "experiment_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved: {summary_csv}")

In [ ]:
# Comparison bar chart: test mIoU per experiment × architecture
if len(summary_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))

    x      = np.arange(len(summary_df))
    labels = [
        f"{r['exp_name'].replace('exp_', '').replace('_deeplabv3plus_cbam', '\nDL').replace('_segformer', '\nSF')}"
        for _, r in summary_df.iterrows()
    ]
    colors = [
        "steelblue" if "deeplabv3" in r["arch"] else "darkorange"
        for _, r in summary_df.iterrows()
    ]

    bars = ax.bar(x, summary_df["test_miou"], color=colors, edgecolor="white", width=0.6)
    for bar, val in zip(bars, summary_df["test_miou"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Test mIoU")
    ax.set_title("Experiment Comparison — Test mIoU (DeepLabV3+CBAM vs SegFormer)")
    ax.set_ylim(0, min(1.0, summary_df["test_miou"].max() + 0.1))
    ax.grid(axis="y", alpha=0.4)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="steelblue",  label="DeepLabV3+CBAM"),
        Patch(color="darkorange", label="SegFormer"),
    ])

    plt.tight_layout()
    chart_path = os.path.join(MODELS_DIR, "experiment_comparison.png")
    plt.savefig(chart_path, dpi=150)
    plt.show()
    print(f"Saved: {chart_path}")

## Training Curves — All Experiments

Load `training_history.csv` from each saved experiment and plot loss + mIoU curves.
Two architectures per experiment configuration are shown on the same axes for direct comparison.

In [ ]:
FIGURES_DIR = os.path.join(PROJECT_ROOT, "documents/thesis/figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

EXP_STYLES = {
    "deeplabv3plus_cbam": {"color": "steelblue",  "label": "DeepLabV3+CBAM"},
    "segformer":          {"color": "darkorange", "label": "SegFormer"},
}
EXP_KEYS = ["A", "B", "C"]

# Load histories from saved CSVs
histories = {}
for exp_key in EXP_KEYS:
    for arch_key, style in EXP_STYLES.items():
        exp_name = f"exp_{exp_key}_{arch_key}"
        csv_path = os.path.join(MODELS_DIR, exp_name, "training_history.csv")
        if os.path.exists(csv_path):
            histories[(exp_key, arch_key)] = pd.read_csv(csv_path)
            print(f"  Loaded: {exp_name}  ({len(histories[(exp_key, arch_key)])} epochs)")
        else:
            print(f"  ⚠️  Missing: {csv_path}")

if not histories:
    print("No training histories found — run experiments first.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    for col, exp_key in enumerate(EXP_KEYS):
        ax_loss = axes[0, col]
        ax_miou = axes[1, col]

        for arch_key, style in EXP_STYLES.items():
            if (exp_key, arch_key) not in histories:
                continue
            df = histories[(exp_key, arch_key)]
            c  = style["color"]
            lbl = style["label"]
            # Train loss (dashed), Val loss (solid)
            ax_loss.plot(df["epoch"], df["train_loss"], color=c, linestyle="--",
                         alpha=0.5, linewidth=1.2)
            ax_loss.plot(df["epoch"], df["val_loss"], color=c, linewidth=2,
                         label=lbl)
            # Val mIoU
            ax_miou.plot(df["epoch"], df["val_miou"], color=c, linewidth=2,
                         label=lbl)
            best_ep = df.loc[df["val_miou"].idxmax(), "epoch"]
            ax_miou.axvline(best_ep, color=c, linestyle=":", alpha=0.5, linewidth=1)

        ax_loss.set(title=f"Exp {exp_key} — Loss", xlabel="Epoch", ylabel="Loss")
        ax_loss.legend(fontsize=8); ax_loss.grid(True, alpha=0.3)

        ax_miou.set(title=f"Exp {exp_key} — Val mIoU", xlabel="Epoch", ylabel="mIoU")
        ax_miou.legend(fontsize=8); ax_miou.grid(True, alpha=0.3)

    # Add a shared note on loss: dashed = train, solid = val
    fig.text(0.5, 1.00, "Training Curves — All Experiments  "
             "(dashed = train loss, solid = val loss)",
             ha="center", fontsize=12, fontweight="bold")
    plt.tight_layout()
    curve_fig = os.path.join(FIGURES_DIR, "training_curves_all.png")
    plt.savefig(curve_fig, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {curve_fig}")


## Test Set Segmentation Visualization

Run full tiled inference on the complete test-year raster and display the assembled
prediction map alongside the CDL ground truth for the whole study area.

In [ ]:
import rasterio
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

S2_NODATA = -9999.0

# Class colours — background + 10 crop classes (index = model sequential class ID)
CROP_COLORS = [
    "#000000",  # 0  background
    "#1E90FF",  # 1  Rice
    "#FFD700",  # 2  Sunflower
    "#8B4513",  # 3  Winter Wheat
    "#98FB98",  # 4  Alfalfa
    "#A9A9A9",  # 5  Other Hay
    "#FF6347",  # 6  Tomatoes
    "#800080",  # 7  Grapes
    "#FF8C00",  # 8  Almonds
    "#228B22",  # 9  Walnuts
    "#9370DB",  # 10 Plums
]
CLASS_LABELS = ["Background"] + list(CLASS_NAMES.values())
SEG_CMAP = ListedColormap(CROP_COLORS)
SEG_NORM = BoundaryNorm(boundaries=range(NUM_CLASSES + 1), ncolors=NUM_CLASSES)

print(f"Colormap: {NUM_CLASSES} classes")
for i, (c, lbl) in enumerate(zip(CROP_COLORS, CLASS_LABELS)):
    print(f"  {i:2d}  {c}  {lbl}")


In [ ]:
def run_full_inference(model, s2_paths, band_indices, patch_size=256, stride=256, device=None):
    """
    Tiled inference over the full raster stack.
    Loads all s2_paths, selects band_indices, slides a window with given stride,
    and assembles the class predictions into a full (H, W) map.
    Returns (pred_map uint8, rasterio profile).
    """
    if device is None:
        device = DEVICE

    arrays, profile = [], None
    for p in s2_paths:
        with rasterio.open(p) as src:
            arr = src.read().astype(np.float32)
            if profile is None:
                profile = dict(src.profile)
        arr[arr == S2_NODATA] = 0.0
        arrays.append(arr)

    stacked  = np.concatenate(arrays, axis=0)   # (C_all, H, W)
    selected = stacked[band_indices]             # (K, H, W)
    K, H, W  = selected.shape
    del stacked

    pred_map = np.zeros((H, W), dtype=np.uint8)
    n_rows   = (H + stride - 1) // stride
    n_cols   = (W + stride - 1) // stride
    total    = n_rows * n_cols

    model.eval()
    done = 0
    with torch.no_grad():
        for y in range(0, H, stride):
            for x in range(0, W, stride):
                ph = min(patch_size, H - y)
                pw = min(patch_size, W - x)
                patch = selected[:, y:y + ph, x:x + pw]

                if ph < patch_size or pw < patch_size:      # pad edge tiles
                    padded = np.zeros((K, patch_size, patch_size), dtype=np.float32)
                    padded[:, :ph, :pw] = patch
                    patch = padded

                t   = torch.from_numpy(patch).unsqueeze(0).to(device)
                out = model(t).argmax(dim=1).squeeze().cpu().numpy()
                pred_map[y:y + ph, x:x + pw] = out[:ph, :pw]
                done += 1
                if done % 100 == 0 or done == total:
                    print(f"  {done}/{total} tiles", end="\r")

    print(f"  {total}/{total} tiles — done{' ' * 20}")
    return pred_map, profile


def load_gt_remap(cdl_path):
    """Load CDL raster and apply REMAP_LUT. Returns (gt_map uint8, profile)."""
    with rasterio.open(cdl_path) as src:
        cdl     = src.read(1).astype(np.int32)
        profile = dict(src.profile)
    gt = REMAP_LUT[np.clip(cdl, 0, 255)]
    return gt.astype(np.uint8), profile


def show_full_maps(pred_map, gt_map, title, save_path, downsample=4):
    """
    Display prediction and GT maps side by side at reduced resolution.
    downsample=4 means every 4th pixel is taken (25% size, faster rendering).
    """
    pred_ds = pred_map[::downsample, ::downsample]
    gt_ds   = gt_map[::downsample, ::downsample]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(gt_ds,   cmap=SEG_CMAP, norm=SEG_NORM, interpolation="nearest")
    axes[0].set_title("Ground Truth (CDL)", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(pred_ds, cmap=SEG_CMAP, norm=SEG_NORM, interpolation="nearest")
    axes[1].set_title("Prediction", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    patches = [mpatches.Patch(color=CROP_COLORS[i], label=CLASS_LABELS[i])
               for i in range(1, NUM_CLASSES)]
    fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, -0.01), frameon=True)

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {save_path}")


# ── Run full-image inference for all 6 experiments ───────────────────────────
def _s2_for_year(yr):
    return sorted([p for p in S2_PROCESSED if os.path.basename(p).split("_")[1] == yr])

test_s2  = _s2_for_year(TEST_YEAR)
test_cdl = CDL_BY_YEAR[TEST_YEAR]

# Load GT once (shared across all experiments)
print("Loading ground truth CDL...")
gt_map, _ = load_gt_remap(test_cdl)
print(f"  GT map shape: {gt_map.shape}")

for exp_name, arch, band_idx, label in [
    ("exp_A_deeplabv3plus_cbam", "deeplabv3plus_cbam", exp_A_idx, "Exp A --- DeepLabV3+CBAM"),
    ("exp_A_segformer",           "segformer",          exp_A_idx, "Exp A --- SegFormer"),
    ("exp_B_deeplabv3plus_cbam", "deeplabv3plus_cbam", exp_B_idx, "Exp B --- DeepLabV3+CBAM"),
    ("exp_B_segformer",           "segformer",          exp_B_idx, "Exp B --- SegFormer"),
    ("exp_C_deeplabv3plus_cbam", "deeplabv3plus_cbam", exp_C_idx, "Exp C --- DeepLabV3+CBAM"),
    ("exp_C_segformer",           "segformer",          exp_C_idx, "Exp C --- SegFormer"),
]:
    if band_idx is None:
        print(f"\n⚠️  Skipping {exp_name} (band indices not set)"); continue

    ckpt_path = os.path.join(MODELS_DIR, exp_name, "best_model.pth")
    if not os.path.exists(ckpt_path):
        print(f"\n⚠️  Skipping {exp_name} (checkpoint not found)"); continue

    print(f"\n{'─'*55}\n {label}  ({len(band_idx)} channels)")
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model = build_model(arch, len(band_idx), NUM_CLASSES)
    model.load_state_dict(ckpt["model_state_dict"])

    # Exp A: pass all test-year files (exp_A_idx uses local offset for July date)
    # Exp B/C: pass all test-year files (exp_B/C_idx use local offsets within per-year list)
    s2_paths = test_s2

    pred_map, _ = run_full_inference(model, s2_paths, band_idx,
                                     patch_size=PATCH_SIZE, stride=PATCH_SIZE)
    save_path = os.path.join(FIGURES_DIR, f"test_segmentation_{exp_name}.png")
    show_full_maps(pred_map, gt_map,
                   title=f"{label} --- Full Test Image Segmentation ({TEST_YEAR})",
                   save_path=save_path)

    del pred_map, model  # free GPU/CPU memory between runs
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\nAll experiments visualized.")
